# Graph Projection

This notebook loads the 20D business-cluster video embedding export from Drive, computes per-channel centroids, builds a channel adjacency graph from relative distances, and projects channels into a 2D layout for plotting.

This cell imports the libraries used for data loading, distance computation, graph construction, and visualization.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS

This cell mounts Google Drive (when running in Colab) and defines the documented export path for the 20D reduced video embeddings artifact.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Colab Drive mount skipped (not running in Colab).')

EMBEDDINGS_CSV = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')
EMBEDDINGS_CSV

This cell loads the 20D video embedding export and validates that the expected reduced embedding columns are present.

In [ ]:
df = pd.read_csv(EMBEDDINGS_CSV)
embedding_cols = [f'embedding_reduced_{i:02d}' for i in range(1, 21)]
missing_cols = [c for c in embedding_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing embedding columns: {missing_cols}')

print(f'Rows: {len(df):,}')
print(f'Channels: {df["channel_id"].nunique():,}')
df[['video_id', 'channel_id', 'video_title'] + embedding_cols[:3]].head()

This cell computes a centroid per channel by averaging each channel's 20D video embeddings.

In [ ]:
channel_centroids = (
    df.groupby('channel_id', as_index=False)[embedding_cols]
      .mean()
)

if 'channel_name' in df.columns:
    channel_labels = (
        df[['channel_id', 'channel_name']]
        .dropna(subset=['channel_id'])
        .drop_duplicates(subset=['channel_id'])
    )
    channel_centroids = channel_centroids.merge(channel_labels, on='channel_id', how='left')
else:
    channel_centroids['channel_name'] = channel_centroids['channel_id']

channel_centroids.head()

This cell builds a relative-distance adjacency matrix between channels using Euclidean distance and row-wise normalization.

In [ ]:
centroid_matrix = channel_centroids[embedding_cols].to_numpy()
channel_ids = channel_centroids['channel_id'].tolist()
channel_names = channel_centroids['channel_name'].fillna(channel_centroids['channel_id']).tolist()

D = pairwise_distances(centroid_matrix, metric='euclidean')
np.fill_diagonal(D, 0.0)

row_sums = D.sum(axis=1, keepdims=True)
A = np.divide(D, row_sums, out=np.zeros_like(D), where=row_sums > 0)

adjacency_df = pd.DataFrame(A, index=channel_ids, columns=channel_ids)
adjacency_df.iloc[:5, :5]

This cell creates a sparse graph by connecting each channel to its k nearest neighboring channels according to centroid distance.

In [ ]:
k = 4
G = nx.Graph()

for i, (cid, cname) in enumerate(zip(channel_ids, channel_names)):
    G.add_node(cid, label=cname)
    neighbor_idx = np.argsort(D[i])[1:k+1]
    for j in neighbor_idx:
        weight = float(D[i, j])
        if i != j:
            G.add_edge(channel_ids[i], channel_ids[j], weight=weight)

print(f'Graph nodes: {G.number_of_nodes()}')
print(f'Graph edges: {G.number_of_edges()}')

This cell projects channels into 2D using metric MDS on the centroid distance matrix and stores the coordinates.

In [ ]:
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
coords_2d = mds.fit_transform(D)

projection_df = pd.DataFrame({
    'channel_id': channel_ids,
    'channel_name': channel_names,
    'x': coords_2d[:, 0],
    'y': coords_2d[:, 1],
})
projection_df.head()

This cell plots the 2D projection and overlays graph edges to visualize the relative channel-distance structure.

In [ ]:
pos = {row.channel_id: (row.x, row.y) for row in projection_df.itertuples(index=False)}

plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.8)
nx.draw_networkx_nodes(G, pos, node_size=70, alpha=0.9)

for row in projection_df.itertuples(index=False):
    plt.text(row.x, row.y, str(row.channel_name), fontsize=8, alpha=0.85)

plt.title('Channel Projection from 20D Video-Embedding Centroids')
plt.xlabel('MDS Dimension 1')
plt.ylabel('MDS Dimension 2')
plt.tight_layout()
plt.show()

This cell saves reusable channel-level outputs (centroids, adjacency matrix, and 2D projection) back to Drive for downstream analysis.

In [ ]:
out_dir = Path('/content/drive/MyDrive/Graphiko/analysis/graph_projection/latest')
out_dir.mkdir(parents=True, exist_ok=True)

channel_centroids.to_csv(out_dir / 'channel_centroids_20d.csv', index=False)
adjacency_df.to_csv(out_dir / 'channel_adjacency_relative_distance.csv')
projection_df.to_csv(out_dir / 'channel_projection_2d.csv', index=False)

summary = {
    'input_csv': str(EMBEDDINGS_CSV),
    'n_videos': int(len(df)),
    'n_channels': int(len(channel_centroids)),
    'embedding_dimensions': 20,
    'graph_edges': int(G.number_of_edges()),
    'projection_method': 'MDS(metric, precomputed euclidean channel-centroid distance)',
}
(out_dir / 'run_summary.json').write_text(json.dumps(summary, indent=2))

print(f'Saved outputs to: {out_dir}')